In [6]:
pip install qiskit qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 103.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 5.5 MB/s eta 0:00:00


In [7]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chisquare
import qiskit
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector, DensityMatrix
from qiskit.visualization import plot_histogram

Exercise 1 Easy -
Shot Count Convergence Analysis

In [9]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
# Construct equal superposition circuit
circuit = QuantumCircuit(1, 1)
circuit.h(0)
circuit.measure(0, 0)
backend = AerSimulator()
shot_settings = [100, 1000, 10000]
print("--- Superposition Circuit ---")
print(circuit)
print(" "+"="*50)
print(f"{'Shot Count':<12}{'P(0)':<12}{'P(1)':<12}{'Delta from 0.5':<15}")
print("="*50)
for count in shot_settings:
    execution = backend.run(circuit, shots=count).result()
    raw_counts = execution.get_counts()
    prob_0 = raw_counts.get('0', 0) / count
    prob_1 = raw_counts.get('1', 0) / count
    deviation = abs(prob_0 - 0.5)
    print(f"{count:<12}{prob_0:<12.4f}{prob_1:<12.4f}{deviation:<15.4f}")
print("="*50)

--- Superposition Circuit ---
     ┌───┐┌─┐
  q: ┤ H ├┤M├
     └───┘└╥┘
c: 1/══════╩═
           0 
Shot Count  P(0)        P(1)        Delta from 0.5 
100         0.5400      0.4600      0.0400         
1000        0.5080      0.4920      0.0080         
10000       0.4993      0.5007      0.0007         


Exercise 2 Medium -
Measurement in the Hadamard (X) Basis

In [10]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
sim = AerSimulator()
test_shots = 1024
# Basis transformation test for |+> and |-> states
circuits = {
    "|+> State in X-Basis": QuantumCircuit(1, 1),
    "|-> State in X-Basis": QuantumCircuit(1, 1)
}
# State preparation & X-basis projection (H before readout)
circuits["|+> State in X-Basis"].h(0)
circuits["|+> State in X-Basis"].h(0)
circuits["|+> State in X-Basis"].measure(0, 0)
circuits["|-> State in X-Basis"].x(0)
circuits["|-> State in X-Basis"].h(0)
circuits["|-> State in X-Basis"].h(0)
circuits["|-> State in X-Basis"].measure(0, 0)
for label, qc in circuits.items():
    res = sim.run(qc, shots=test_shots).result().get_counts()
    print(f"--- {label} ---")
    print(qc)
    print(f"Measured Outcomes: {res}")

--- |+> State in X-Basis ---
     ┌───┐┌───┐┌─┐
  q: ┤ H ├┤ H ├┤M├
     └───┘└───┘└╥┘
c: 1/═══════════╩═
                0 
Measured Outcomes: {'0': 1024}
--- |-> State in X-Basis ---
     ┌───┐┌───┐┌───┐┌─┐
  q: ┤ X ├┤ H ├┤ H ├┤M├
     └───┘└───┘└───┘└╥┘
c: 1/════════════════╩═
                     0 
Measured Outcomes: {'1': 1024}


Exercise 3 Hard -
Partial Measurement on a Bell State

In [11]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, DensityMatrix
# Build Bell State (|00> + |11>) / sqrt(2)
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
initial_psi = Statevector.from_instruction(bell)
print("--- Bell State Circuit ---")
print(bell)
print(" Initial Statevector:", np.round(initial_psi.data, 4))
# Partial measurement performed on target qubit index 0
bit_measured, collapsed_psi = initial_psi.measure([0])
print(f"Target Qubit 0 Measurement Outcome: '{bit_measured}'")
print("Post-Measurement Collapsed Statevector:")
print(np.round(collapsed_psi.data, 4))
# Analyze reduced density matrix of qubit 1
rho_qubit1 = DensityMatrix(collapsed_psi).to_dict()
print("Density Matrix of Qubit 1:", rho_qubit1)

--- Bell State Circuit ---
     ┌───┐     
q_0: ┤ H ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘
 Initial Statevector: [0.7071+0.j 0.    +0.j 0.    +0.j 0.7071+0.j]
Target Qubit 0 Measurement Outcome: '1'
Post-Measurement Collapsed Statevector:
[0.+0.j 0.+0.j 0.+0.j 1.+0.j]
Density Matrix of Qubit 1: {'11|11': np.complex128(1+0j)}


Exercise 4 Real-world -
Quantum State Parameter Estimation

In [12]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
# Target ground-truth rotation angle
actual_theta = 0.9273  # rad
total_samples = 8000
aer = AerSimulator()
# Prepare state via Ry rotation and measure along Z-basis
tomo_circ = QuantumCircuit(1, 1)
tomo_circ.ry(actual_theta, 0)
tomo_circ.measure(0, 0)
exp_counts = aer.run(tomo_circ, shots=total_samples).result().get_counts()
freq_0 = exp_counts.get('0', 0) / total_samples
# Tomographic parameter reconstruction: P(0) = cos^2(theta / 2)
reconstructed_theta = 2.0 * np.arccos(np.sqrt(freq_0))
abs_discrepancy = abs(actual_theta - reconstructed_theta)
print("--- State Tomography Circuit ---")
print(tomo_circ)
print(f"Measurement Distribution: {exp_counts}")
print(f"Empirical P(0):         {freq_0:.4f}")
print(f"Ground-Truth Theta:     {actual_theta:.4f} rad")
print(f"Reconstructed Theta:    {reconstructed_theta:.4f} rad")
print(f"Estimation Error:       {abs_discrepancy:.4f} rad")

--- State Tomography Circuit ---
     ┌────────────┐┌─┐
  q: ┤ Ry(0.9273) ├┤M├
     └────────────┘└╥┘
c: 1/═══════════════╩═
                    0 
Measurement Distribution: {'1': 1595, '0': 6405}
Empirical P(0):         0.8006
Ground-Truth Theta:     0.9273 rad
Reconstructed Theta:    0.9257 rad
Estimation Error:       0.0016 rad


Exercise 5 Challenge -
Quantum Randomness vs. Classical PRNG

In [13]:
import numpy as np
from scipy.stats import chisquare
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
sample_count = 3000
sim_backend = AerSimulator()
# 1. Quantum Random Bit Generator
q_circ = QuantumCircuit(1, 1)
q_circ.h(0)
q_circ.measure(0, 0)
q_results = sim_backend.run(q_circ, shots=sample_count).result().get_counts()
qrng_data = [q_results.get('0', 0), q_results.get('1', 0)]
# 2. Biased Classical Pseudo-Random Number Generator (P(0) = 0.535)
classical_stream = np.random.choice([0, 1], size=sample_count, p=[0.535, 0.465])
prng_data = [int(np.sum(classical_stream == 0)), int(np.sum(classical_stream == 1))]
# Expected uniform frequencies
uniform_expected = [sample_count / 2.0, sample_count / 2.0]
stat_q, pval_q = chisquare(f_obs=qrng_data, f_exp=uniform_expected)
stat_c, pval_c = chisquare(f_obs=prng_data, f_exp=uniform_expected)
print("--- Goodness-of-Fit Hypothesis Test (alpha = 0.05) ---")
print(f"QRNG Data [0, 1]:  {qrng_data} | Chi2: {stat_q:.4f} | p-val: {pval_q:.4f}")
print(f"QRNG Conclusion:   {'Accept H0: True Uniform Random' if pval_q > 0.05 else 'Reject H0: Biased'}")
print(f"PRNG Data [0, 1]:  {prng_data} | Chi2: {stat_c:.4f} | p-val: {pval_c:.4f}")
print(f"PRNG Conclusion:   {'Accept H0: True Uniform Random' if pval_c > 0.05 else 'Reject H0: Biased'}")

--- Goodness-of-Fit Hypothesis Test (alpha = 0.05) ---
QRNG Data [0, 1]:  [1485, 1515] | Chi2: 0.3000 | p-val: 0.5839
QRNG Conclusion:   Accept H0: True Uniform Random
PRNG Data [0, 1]:  [1574, 1426] | Chi2: 7.3013 | p-val: 0.0069
PRNG Conclusion:   Reject H0: Biased
